### Harri and Valtteri's data 

In [11]:
import pandas as pd 
df = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="SSGDD study")
#print(df)
# also run the topics through LLM2. 
# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
import torch
import numpy as np
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")

# load model from a working checkpoint 
model_name='BERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 
# misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=False)


In [45]:
# specifically for DD data. 
df2 = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="Sara Mappings")
df2['S'] = df2['KAs (mapping)']
df3 = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="Paul Mappings")
df2['P'] = df3['KAs (mapping)']
df4 = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="Valtteri Mappings")
df2['V'] = df4['KAs (mapping)']
df2['DD'] = ['' for x in range(len(df2))]
print(df2.keys())
#print(df2['KD ID'])

Index(['KD ID', 'KD', 'KAs (mapping)', 'Reasonings', 'S', 'P', 'V', 'DD'], dtype='object')


In [47]:
# label with LLM 
import re
from datasets import Dataset 
from utils.load_data import clean_text
device = 'cuda:0'
KA_labels = [] 
course_descriptions = df2.get(df2.keys()[1])
print(course_descriptions[0])

# make them into the right format (a dataset?) 
th = 0.5 
llm_labels = [] 
llm_labelsN = [] 
tokenized_course = {}
for i in range(len(course_descriptions)): 
    #print(course_descriptions[i][13:])
    tokenized_course[str(i)] = tokenizer(course_descriptions[i][13:], return_tensors='pt') #, padding='max_length', max_length=64)
    with torch.no_grad(): 
        predictions= model(**tokenized_course[str(i)])
        prediction = sigmoid(predictions.logits).detach().cpu().numpy()
        predictions = (prediction > th).astype(int).reshape(-1)
        
        if sum(predictions) == 0: 
            print('empty')
            out = np.argmax(prediction)
            prediction_zeros= np.zeros(predictions.shape)
            prediction_zeros[out] = 1 
            predictions = prediction_zeros.astype(int)
        llm_labels.append(predictions)
        print(predictions)
        #out = [idx for idx,x in enumerate(predictions) if x >0]
        llm_labelsN.append([idx for idx,x in enumerate(predictions) if x >0])
        print([idx for idx,x in enumerate(predictions) if x >0])
# combine all labels of the course. 
llm_labels2 = np.array(llm_labels)
#print(llm_labels2)
final_output = np.sum(llm_labels2,axis=0) / (np.sum(llm_labels2)) #/ len(course.features) # divide by the total number of 'items' 

#print('LLM Labels: ',final_output)
KA_labels.append(final_output) 
#print(llm_labelsN)
row_data = [','.join(map(str, sublist)) for sublist in llm_labelsN]
#print(row_data)
row_data2 = []
for x in row_data: 
    if len(x) == 0: 
        row_data2.append('NAN')
    else: 
        row_data2.append(x)
#row_data2 = ['NAN' for x in row_data if len(x)==0 else x]
print(row_data2)
df2['LLM'] = row_data2
#print(df)
print(df2['LLM'][:])

Knowledge of the organizational cybersecurity workforce
[0 0 0 0 0 0 0 1 0]
[7]
[0 0 0 0 1 0 0 0 0]
[4]
[0 0 0 0 1 0 0 0 0]
[4]
[0 0 0 0 0 1 0 0 0]
[5]
[1 0 0 0 0 0 0 0 0]
[0]
[0 0 0 0 0 0 0 1 0]
[7]
[0 0 0 0 1 0 0 0 0]
[4]
[1 0 0 0 0 0 0 0 0]
[0]
[0 0 0 0 1 0 0 0 0]
[4]
[1 0 0 0 0 0 0 0 0]
[0]
[0 0 0 0 0 0 0 1 0]
[7]
[0 0 1 1 0 0 0 0 0]
[2, 3]
[1 0 0 0 0 0 0 0 0]
[0]
[0 0 0 0 0 0 0 1 0]
[7]
[1 0 0 0 0 0 0 0 0]
[0]
[0 0 0 0 0 0 0 1 0]
[7]
[1 0 0 0 0 0 0 0 0]
[0]
[1 0 0 0 0 0 0 0 0]
[0]
[0 0 1 0 0 0 0 0 0]
[2]
[0 0 0 0 0 0 0 1 0]
[7]
[0 0 0 0 0 0 0 1 0]
[7]
[0 0 0 0 0 0 0 1 0]
[7]
[1 0 0 0 0 0 0 0 0]
[0]
[1 0 0 0 0 0 0 0 0]
[0]
[0 0 0 0 0 0 0 1 0]
[7]
[0 1 0 0 0 0 1 0 0]
[1, 6]
[0 0 0 0 0 1 0 0 0]
[5]
[1 0 0 0 0 0 0 0 0]
[0]
[1 0 0 0 0 0 0 0 0]
[0]
empty
[1 0 0 0 0 0 0 0 0]
[0]
[1 0 0 0 0 0 0 0 0]
[0]
empty
[0 0 1 0 0 0 0 0 0]
[2]
[0 0 0 0 0 0 0 1 0]
[7]
[0 1 0 0 0 0 0 0 0]
[1]
[0 0 0 0 1 0 0 0 0]
[4]
[0 0 0 0 0 0 0 1 0]
[7]
[0 0 0 0 1 0 0 0 0]
[4]
[0 1 0 0 0 0 1 0 0]
[1, 6]
[0 1 0 0 0 

### Check overlap between S, V, P and align it with the old data 

In [18]:
print(df2.head())

   KD ID                                                 KD  KAs (mapping)  \
0  K0640  Knowledge of the organizational cybersecurity ...              7   
1  K0643         Knowledge of virtual learning environments              0   
2  K0645  Knowledge of standard operating procedures (SOPs)              7   
3  K0662          Knowledge of systems security engineering              4   
4  K0706  Knowledge of database schema capabilities and ...              0   

                                          Reasonings  S    P  V DD LLM  
0                                                NaN  7  7.6  7      7  
1                                           pedagogy  0    0  6      4  
2  A standard operating procedure is a set of ste...  7  7.5  7      4  
3                            Systems fall under KA-4  4    5  5      5  
4                                                NaN  0  1.2  0      0  


In [31]:
# compute both LLM correspondence 
from itertools import combinations
S_data = [str(x) for x in df2['S'].tolist()]
V_data = [str(x) for x in df2['V'].tolist()]
P_data = [str(x) for x in df2['P'].tolist()]
LLM_data = [str(x) for x in df2['LLM'].tolist()]

from nltk import agreement
from nltk.metrics.distance import masi_distance
from nltk.metrics.distance import jaccard_distance

def create_annot(an):
    """
    Create frozensets from comma-separated labels or single labels.
    Handles empty values and converts all labels to strings.
    """
    if pd.isna(an) or an == '':
        return frozenset()
    an = str(an)
    if "," in an:
        # Split on commas and strip whitespace from each label
        return frozenset(label.strip() for label in an.split(","))
    else:
        # Single label
        return frozenset([an.strip()])


def blub(list1, list2): 
    def digits(s): 
        return set(filter(str.isdigit, s)) 
    count = 0 
    for a,b in zip(list1, list2): 
        if digits(a) & digits(b): 
            count+=1 
    return count 
    
def blub2(df, row1, row2):
    annots = []
    for idx, row in df.iterrows():
        annot_id = str.zfill(str(idx), 3)
        annot_coder2 = ['P', annot_id, create_annot(str(row[row1]))]
        annot_coder3 = ['LLM', annot_id, create_annot(str(row[row2]))]
        annots.append(annot_coder2)
        annots.append(annot_coder3)
    task = agreement.AnnotationTask(distance=jaccard_distance)
    task.load_array(annots)
    kappa = task.kappa() 
    return kappa 

# Create a list of (name, data) pairs
data_lists = [("S", S_data,4), ("V", V_data,6), ("P", P_data,5), ("LLM",LLM_data,8)]

for (name1, i, ii), (name2, j,jj) in combinations(data_lists, 2):
    overlap = blub(i, j) 
    print(f"Combination {name1} & {name2}: {overlap/len(S_data)}")
    kappa = blub2(df2, ii, jj)
    print("Cohen's Kappa: {}".format(kappa))

Combination S & V: 0.52
Cohen's Kappa: 0.358302776322682
Combination S & P: 0.54
Cohen's Kappa: 0.335548172757475
Combination S & LLM: 0.56
Cohen's Kappa: 0.39421573736321003
Combination V & P: 0.56
Cohen's Kappa: 0.3350443303779748
Combination V & LLM: 0.56
Cohen's Kappa: 0.3861607142857143
Combination P & LLM: 0.54
Cohen's Kappa: 0.2740046838407494


### DD data analysis

In [40]:
import pandas as pd
# Get all unique KD values
df = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="SSGDD study")
unique_kds = pd.unique(df[['First KD', 'Second KD', 'Third KD']].values.ravel('K'))
# Initialize a list to store results
results = []
for kd in unique_kds:
    mappings_list = [] # Collect all mappings for this KD (as strings, without splitting)
    first_kd_rows = df[df['First KD'] == kd] # Check First KD column
    mappings_list.extend(first_kd_rows['First Mappings'].dropna().astype(str).tolist())
    second_kd_rows = df[df['Second KD'] == kd] # Check Second KD column
    mappings_list.extend(second_kd_rows['Second Mappings'].dropna().astype(str).tolist())
    third_kd_rows = df[df['Third KD'] == kd] # Check Third KD column
    mappings_list.extend(third_kd_rows['Third Mappings'].dropna().astype(str).tolist())
    
    # Create a dictionary for this KD
    kd_data = {'KD': kd}
    # Add each occurrence as a new column
    for i, mapping in enumerate(mappings_list, start=1):
        kd_data[f'occurrence_{i}'] = mapping
    results.append(kd_data)

result_df = pd.DataFrame(results) # Convert to DataFrame
result_df = result_df.fillna('') # Fill NaN for missing occurrences (some KDs may have fewer mappings)
data = result_df.drop([result_df.columns[0], result_df.columns[6],result_df.columns[7],result_df.columns[8]], axis=1) # Minimum # occurrences = 5, so cut it off at 5 annotators. 
#print(data)
print(np.sort(unique_kds))

['K0645' 'K0710' 'K0713' 'K0757' 'K0775' 'K0798' 'K0807' 'K0829' 'K0941'
 'K1126' 'K1145' 'K1193' 'K1207' 'K1248' 'K1279']


In [57]:
print(result_df)

       KD               occurrence_1      occurrence_2         occurrence_3  \
0   K1248                          5           2, 3, 5        1, 3, 4, 6, 8   
1   K1145                    1, 3, 4                 5                    1   
2   K1279                          4                 7     6, 5, 4, 3, 1, 2   
3   K0713                          5  1, 3, 4, 5, 7, 0              4, 5, 7   
4   K0798                    6, 7, 8     5, 6, 7, 8, 0                    5   
5   K0829                    3, 7, 8           7, 6, 0                    5   
6   K1207              2, 4, 5, 1, 0                 2  1, 3, 4, 5, 6, 8, 0   
7   K0645  1, 2, 3, 4, 5, 6, 7, 8, 0              6, 7     8, 6, 7, 5, 3, 2   
8   K0775              1, 2, 4, 5, 6           7, 8, 3              7, 6, 0   
9   K0807                       1, 2                 1                 4, 6   
10  K0757                       2, 5                 5  6, 5, 4, 3, 2, 1, 0   
11  K1193                       2, 4     1, 2, 3, 4,

In [51]:
# just for the DD 
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
S_data = [str(x) for x in df2['S'].tolist()]
V_data = [str(x) for x in df2['V'].tolist()]
P_data = [str(x) for x in df2['P'].tolist()]
LLM_data = [str(x) for x in df2['LLM'].tolist()]

data['S'], data['P'], data['V'], data['LLM'] = [['' for i in range(len(data))] for ii in range(4)]
for i in range(len(df2)): 
    for j in range(len(data)): 
        if df2['KD ID'][i] == unique_kds[j]: 
            df2['DD'][i] = data['occurrence_3'][j]
            data['S'][j] = df2['S'][i]
            data['P'][j] = df2['P'][i]
            data['V'][j] = df2['V'][i]
            data['LLM'][j] = df2['LLM'][i]
print(data)

                 occurrence_1      occurrence_2         occurrence_3  \
0                           5           2, 3, 5        1, 3, 4, 6, 8   
1                     1, 3, 4                 5                    1   
2                           4                 7     6, 5, 4, 3, 1, 2   
3                           5  1, 3, 4, 5, 7, 0              4, 5, 7   
4                     6, 7, 8     5, 6, 7, 8, 0                    5   
5                     3, 7, 8           7, 6, 0                    5   
6               2, 4, 5, 1, 0                 2  1, 3, 4, 5, 6, 8, 0   
7   1, 2, 3, 4, 5, 6, 7, 8, 0              6, 7     8, 6, 7, 5, 3, 2   
8               1, 2, 4, 5, 6           7, 8, 3              7, 6, 0   
9                        1, 2                 1                 4, 6   
10                       2, 5                 5  6, 5, 4, 3, 2, 1, 0   
11                       2, 4     1, 2, 3, 4, 5                    3   
12                 1, 2, 3, 4     1, 2, 3, 5, 6        1, 2, 3, 

In [56]:
S_data = [str(x) for x in data['S'].tolist()]
V_data = [str(x) for x in data['V'].tolist()]
P_data = [str(x) for x in data['P'].tolist()]
LLM_data = [str(x) for x in data['LLM'].tolist()]
for ii in [("S", S_data,5), ("V", V_data,7), ("P", P_data,6), ("LLM",LLM_data,8)]: 
    ann1 = [str(x) for x in data['occurrence_1'].tolist()]
    ann2 = [str(x) for x in data['occurrence_2'].tolist()]
    ann3 = [str(x) for x in data['occurrence_3'].tolist()]
    ann4 = [str(x) for x in data['occurrence_4'].tolist()]
    ann5 = [str(x) for x in data['occurrence_5'].tolist()]
    data_lists = [("ann1",ann1,0),("ann2",ann2,1), ("ann3",ann3,2),("ann4", ann4,3),("ann5", ann5,4), ii]
    sums = [] 
    kappas = [] 
    idx =0 
    for (name1, i, ii), (name2, j,jj) in combinations(data_lists, 2):
        overlap = blub(i, j) 
        kappa = blub2(data, ii, jj)
        if idx == 4 or idx==8 or idx==11 or idx==13 or idx==14: 
           # print(name1+name2)
           # print(i)
           # print(len(j))
           # print(overlap)
            sums.append(100*overlap/len(S_data))
            kappas.append(kappa)
        idx += 1 
    print(f"Combination DD & {name2}")
    print('score: ',np.mean(sums))
    print('kappas: ',np.mean(kappas))

Combination DD & S
score:  44.0
kappas:  0.1472149980243722
Combination DD & V
score:  60.0
kappas:  0.20544046399884527
Combination DD & P
score:  68.0
kappas:  0.2095634861896804
Combination DD & LLM
score:  60.0
kappas:  0.21539751573204904


### Old code 

In [52]:
# Merge DD annotator 1, 2, 3, 4, and 5
import re 
from collections import Counter
def return_most_frequent(labels_combined): 
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', labels_combined) 
    counter = Counter(numbers)
    max_frequency = max(counter.values())
    most_frequent = [num for num, count in counter.items() if count == max_frequency]
    # format the output as a string 
    if len(most_frequent) == 1: 
        most_frequent = most_frequent[0]
    else: 
        most_frequent = ",".join(most_frequent)
    return most_frequent

# merge annotator 1, 2, and 3 using the label that occurs most often 
merged = [] 
ann1_data = data['occurrence_1']
ann2_data = data['occurrence_2']
ann3_data = data['occurrence_3']
ann4_data = data['occurrence_4']
ann5_data = data['occurrence_5']
for i,ele in enumerate(ann1_data): 
    # leave out any NAN 
    if ann1_data[i] == 'NAN': 
        ann1 = ''
    else: 
        ann1 = str(ann1_data[i])
    if ann2_data[i] == 'NAN': 
        ann2 = ''
    else: 
        ann2 = str(ann2_data[i])
    if ann3_data[i] =='NAN': 
        ann3 = ''
    else: 
        ann3 = str(ann3_data[i])
    if ann4_data[i] =='NAN': 
        ann4 = ''
    else: 
        ann4 = str(ann4_data[i])
    if ann5_data[i] =='NAN': 
        ann5 = ''
    else: 
        ann5 = str(ann5_data[i])
    # combine labels 
    labels_combined = ann1+',' + ann2 +','+ ann3 #+','+ ann4 +','+ ann5
    # choose the maximum only, if multiple have the same number of occurrences, choose both 
    most_frequent = return_most_frequent(labels_combined)
    #print(most_frequent)
    merged.append(most_frequent)

data['merged'] = merged
print(data['merged'])

for i in range(len(df2)): 
    for j in range(len(unique_kds)): 
        if df['KD ID'][i] == unique_kds[j]: 
            print(df2['KD ID'][i])
            print(data['merged'][j])
            df['DD'][i] = data['merged'][j]
print(df)

0           5,3
1             1
2             4
3             5
4       6,7,8,5
5             7
6     2,4,5,1,0
7           6,7
8           6,7
9             1
10            5
11        2,4,3
12        1,2,3
13            7
14            0
Name: merged, dtype: object
K0645
6,7
K0710
7
K0713
5
K0757
5
K0775
6,7
K0798
6,7,8,5
K0807
1
K0829
7
K0941
1,2,3
K1126
0
K1145
1
K1193
2,4,3
K1207
2,4,5,1,0
K1248
5,3
K1279
4
    KD ID                                                 KD  KAs (mapping)  \
0   K0640  Knowledge of the organizational cybersecurity ...              7   
1   K0643         Knowledge of virtual learning environments              0   
2   K0645  Knowledge of standard operating procedures (SOPs)              7   
3   K0662          Knowledge of systems security engineering              4   
4   K0706  Knowledge of database schema capabilities and ...              0   
5   K0710  Knowledge of enterprise cybersecurity architec...              7   
6   K0713              Knowledg

/tmp/ipykernel_2208745/2892650108.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DD'][i] = data['merged'][j]
/tmp/ipykernel_2208745/2892650108.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DD'][i] = data['merged'][j]
/tmp/ipykernel_2208745/2892650108.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DD'][i] = data['merged'][j]
/tmp/ipykernel_2208745/2892650108.py:60: SettingWithCopyWarning: 
A value is tryin

In [66]:
# Krippendorff's Alpha 
import numpy as np
import pandas as pd

def parse_labels(label_str):
    """Convert a comma-separated string of labels into a set of integers."""
    if label_str == 'NAN' or pd.isna(label_str):
        return set()
    label_str = str(label_str)
    return set(map(int, label_str.split(',')))

def multi_hot_matrix(df, label_set):
    """Convert dataframe of annotator columns into a list of multi-hot matrices per annotator."""
    matrices = []
    for col in df.columns:
        encoded = []
        for label_str in df[col]:
            labels = parse_labels(label_str)
            row = [1 if l in labels else 0 for l in label_set]
            encoded.append(row)
        matrices.append(np.array(encoded))
    return np.stack(matrices, axis=1)  # shape: (n_items, n_annotators, n_labels)

def krippendorffs_alpha(data_matrix):
    """
    Computes Krippendorff's Alpha for multi-label data.
    
    data_matrix: numpy array (n_items, n_annotators, n_labels)
    """
    n_items, n_annotators, n_labels = data_matrix.shape
    
    # Step 1: Calculate observed disagreement
    observed_disagreements = np.zeros((n_items, n_labels))
    for i in range(n_items):
        for l in range(n_labels):
            obs = np.sum(data_matrix[i, :, l])  # How many annotators said "1" for this label
            observed_disagreements[i, l] = (n_annotators - obs) / n_annotators

    # Step 2: Calculate expected disagreement (chance agreement)
    expected_disagreements = np.mean(observed_disagreements, axis=0)
    
    # Step 3: Calculate Krippendorff's alpha using the formula
    D_observed = np.mean(observed_disagreements)
    D_expected = np.mean(expected_disagreements)
    
    alpha = 1 - D_observed / D_expected
    return alpha

# Example dataset (same as previous)
#df = pd.DataFrame({
#    'ann1': ['1,2', '2,5', 'NAN', '4,5,6'],
 #   'ann2': ['1,2', '5', '3', '4,6'],
 #   'ann3': ['2', '2,5,6', 'NAN', '4,5,6']
#})

# All unique labels used
all_labels = sorted(set().union(*map(parse_labels, data.get(data.keys()[0])))
                    .union(*map(parse_labels, data.get(data.keys()[1])))
                    .union(*map(parse_labels, data.get(data.keys()[2]))))
print(all_labels)
# Create (n_items, n_annotators, n_labels) multi-hot matrix
multi_hot_data = multi_hot_matrix(data, all_labels)
#print(multi_hot_data)
# Compute Krippendorff's Alpha
alpha = krippendorffs_alpha(multi_hot_data)
print(f"Krippendorff's Alpha: {alpha:.3f}")

[0, 1, 2, 3, 4, 5, 6, 7, 8]
Krippendorff's Alpha: -0.000


In [69]:
#print(result_df)
# merge result_df 
# find sara and paul and valtteri's annotations 
#df = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="Sara Mappings")
#df['S'] = df['KAs (mapping)']
#df2 = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="Paul Mappings")
#df['P'] = df2['KAs (mapping)']
#df3 = pd.read_excel('data/new_random_50_kd_mappings_with_AI_mappings2.xlsx',sheet_name="Valtteri Mappings")
#df['V'] = df3['KAs (mapping)']
print(df.keys())

Index(['KD ID', 'KD', 'KAs (mapping)', 'Reasonings', 'S', 'P', 'V', 'DD'], dtype='object')


In [74]:
def count_positional_overlaps(lists_of_strings, list_names):
    """
    Count how many positions have overlapping numbers between pairs of lists
    
    Args:
        lists_of_strings: List of 4 lists containing strings in order: 
                         llm_data, S_data, V_data, swits_data
    """
   
    
    # Verify all lists have the same length
    list_lengths = [len(lst) for lst in lists_of_strings]
    if len(set(list_lengths)) > 1:
        raise ValueError("All lists must have the same length")
    
    n_samples = len(lists_of_strings[0])
    
    # Precompute number sets for all positions
    number_sets = []
    for lst in lists_of_strings:
        list_sets = [extract_numbers(s) for s in lst]
        number_sets.append(list_sets)
    
    results = {}
    
    # Check all pair-wise combinations of lists
    for i, j in combinations(range(len(lists_of_strings)), 2):
        count = 0
        for pos in range(n_samples):
            set_i = number_sets[i][pos]
            set_j = number_sets[j][pos]
            
            # Check if both strings have numbers and there's overlap
            if set_i and set_j and set_i.intersection(set_j):
                count += 1
        
        pair_name = f"{list_names[i]}_vs_{list_names[j]}"
        results[pair_name] = count
    
    return results

for ii in ['LLM', 'Sara ', 'Valtteri', 'Paul']:
    print(ii)
    list_names = ['Annotator 1', 'Annotator 2','Annotator 3', ii]
    
    llm_data = [str(x) for x in df['Annotator 1'].tolist()]
    S_data = [str(x) for x in df['Annotator 2 '].tolist()]
    V_data = [str(x) for x in df['Annotator 3 '].tolist()]
    swits_data = [str(x) for x in df[ii].tolist()]
    lists = [llm_data, S_data, V_data, swits_data]
    result = count_positional_overlaps(lists, list_names)
    
    print("Average position-wise overlapping counts, SWITS:")
    #print("=" * 50)
    sums = [] 
    for i, (pair, count) in enumerate(result.items()):
       # print(f"{pair}: {100*count/len(llm_data)} positions")
        if i == 2 or i==4 or i==5: 
            sums.append(100*count/len(llm_data))
    print('score: ',np.mean(sums))

LLM


KeyError: 'Annotator 1'

In [75]:
from nltk import agreement
from nltk.metrics.distance import masi_distance
from nltk.metrics.distance import jaccard_distance

def create_annot(an):
    """
    Create frozensets from comma-separated labels or single labels.
    Handles empty values and converts all labels to strings.
    """
    if pd.isna(an) or an == '':
        return frozenset()
    an = str(an)
    if "," in an:
        # Split on commas and strip whitespace from each label
        return frozenset(label.strip() for label in an.split(","))
    else:
        # Single label
        return frozenset([an.strip()])
annots = []
for idx, row in df.iterrows():
    annot_id = str.zfill(str(idx), 3)
    #annot_coder1 = ['S', annot_id, create_annot(str(row[4]))]
    annot_coder2 = ['P', annot_id, create_annot(str(row[5]))]
  
    annot_coder3 = ['LLM', annot_id, create_annot(str(row[8]))]
    #print(annot_coder3)
    #annots.append(annot_coder1)
    if len(annot_coder3[2] ) !=0: 
        #annots.append(annot_coder1)
        annots.append(annot_coder2)
        annots.append(annot_coder3)

# based on https://stackoverflow.com/questions/45741934/
task = agreement.AnnotationTask(distance=jaccard_distance)
print(annot_coder3)
task.load_array(annots)
print("P-LLM")
print("Cohen's Kappa: {}".format(task.kappa()))
print("Krippendorff's Alpha: {}".format(task.alpha()))

annots = []
for idx, row in df.iterrows():
    annot_id = str.zfill(str(idx), 3)
    #annot_coder1 = ['S', annot_id, create_annot(str(row[4]))]
    annot_coder2 = ['V', annot_id, create_annot(str(row[6]))]
    annot_coder3 = ['LLM', annot_id, create_annot(str(row[8]))]
    if len(annot_coder3[2] ) !=0: 
        #annots.append(annot_coder1)
    #annots.append(annot_coder1)
        annots.append(annot_coder2)
        annots.append(annot_coder3)

# based on https://stackoverflow.com/questions/45741934/
task = agreement.AnnotationTask(distance=jaccard_distance)

task.load_array(annots)
print("V-LLM")
print("Cohen's Kappa: {}".format(task.kappa()))
print("Krippendorff's Alpha: {}".format(task.alpha()))

annots = []
for idx, row in df.iterrows():
    annot_id = str.zfill(str(idx), 3)
    #annot_coder1 = ['S', annot_id, create_annot(str(row[4]))]
    annot_coder2 = ['S', annot_id, create_annot(str(row[4]))]
    annot_coder3 = ['LLM', annot_id, create_annot(str(row[8]))]
    if len(annot_coder3[2] ) !=0: 
        #annots.append(annot_coder1)
    #annots.append(annot_coder1)
        annots.append(annot_coder2)
        annots.append(annot_coder3)

# based on https://stackoverflow.com/questions/45741934/
task = agreement.AnnotationTask(distance=jaccard_distance)

task.load_array(annots)
print("S-LLM")
print("Cohen's Kappa: {}".format(task.kappa()))
print("Krippendorff's Alpha: {}".format(task.alpha()))

annots = []
for idx, row in df.iterrows():
    annot_id = str.zfill(str(idx), 3)
    #annot_coder1 = ['S', annot_id, create_annot(str(row[4]))]
    annot_coder2 = ['S', annot_id, create_annot(str(row[8]))]
    annot_coder3 = ['LLM', annot_id, create_annot(str(row[7]))]
    if len(annot_coder3[2] ) !=0: 
        #annots.append(annot_coder1)
        annots.append(annot_coder2)
        annots.append(annot_coder3)

# based on https://stackoverflow.com/questions/45741934/
task = agreement.AnnotationTask(distance=jaccard_distance)

task.load_array(annots)
print("LLM-DD")
print("Cohen's Kappa: {}".format(task.kappa()))
print("Krippendorff's Alpha: {}".format(task.alpha()))

annots = []
for idx, row in df.iterrows():
    annot_id = str.zfill(str(idx), 3)
    #annot_coder1 = ['S', annot_id, create_annot(str(row[4]))]
    annot_coder2 = ['P', annot_id, create_annot(str(row[5]))]
    annot_coder3 = ['V', annot_id, create_annot(str(row[6]))]
    #annots.append(annot_coder1)
    annots.append(annot_coder2)
    annots.append(annot_coder3)

# based on https://stackoverflow.com/questions/45741934/
task = agreement.AnnotationTask(distance=jaccard_distance)

task.load_array(annots)
print("PV")
print("Cohen's Kappa: {}".format(task.kappa()))
print("Krippendorff's Alpha: {}".format(task.alpha()))

annots = []
for idx, row in df.iterrows():
    annot_id = str.zfill(str(idx), 3)
    annot_coder1 = ['S', annot_id, create_annot(str(row[4]))]
    #annot_coder2 = ['P', annot_id, create_annot(str(row[5]))]
    annot_coder3 = ['V', annot_id, create_annot(str(row[6]))]
 
    annots.append(annot_coder1)
    #annots.append(annot_coder2)
    annots.append(annot_coder3)

# based on https://stackoverflow.com/questions/45741934/
task = agreement.AnnotationTask(distance=jaccard_distance)

task.load_array(annots)
print("SV")
print("Cohen's Kappa: {}".format(task.kappa()))
print("Krippendorff's Alpha: {}".format(task.alpha()))

annots = []
for idx, row in df.iterrows():
    annot_id = str.zfill(str(idx), 3)
    annot_coder1 = ['S', annot_id, create_annot(str(row[4]))]
    annot_coder2 = ['P', annot_id, create_annot(str(row[5]))]
    #annot_coder3 = ['V', annot_id, create_annot(str(row[6]))]
    annots.append(annot_coder1)
    annots.append(annot_coder2)
    #annots.append(annot_coder3)

# based on https://stackoverflow.com/questions/45741934/
task = agreement.AnnotationTask(distance=jaccard_distance)

task.load_array(annots)
print("SP")
print("Cohen's Kappa: {}".format(task.kappa()))
print("Krippendorff's Alpha: {}".format(task.alpha()))


['LLM', '049', frozenset({'7'})]
P-LLM
Cohen's Kappa: 0.2740046838407494
Krippendorff's Alpha: 0.2591400488434875
V-LLM
Cohen's Kappa: 0.3861607142857143
Krippendorff's Alpha: 0.37093233719721186
S-LLM
Cohen's Kappa: 0.39421573736321003
Krippendorff's Alpha: 0.3810836246302769
LLM-DD


ZeroDivisionError: division by zero

P-AI
Cohen's Kappa: 0.28030303030303033
Krippendorff's Alpha: 0.24544196401708906
V-AI
Cohen's Kappa: 0.3697645013434488
Krippendorff's Alpha: 0.327202075850345
S-AI
Cohen's Kappa: 0.4262092793682133
Krippendorff's Alpha: 0.3849178955244489
PV
Cohen's Kappa: 0.3350443303779748
Krippendorff's Alpha: 0.32077243128081845
SV
Cohen's Kappa: 0.358302776322682
Krippendorff's Alpha: 0.3473698372124311
SP
Cohen's Kappa: 0.335548172757475
Krippendorff's Alpha: 0.31336283770394346


In [78]:
#print(df.head())
ann1_data = data['occurrence_1']
ann2_data = data['occurrence_2']
ann3_data = data['occurrence_3']
ann4_data = data['occurrence_4']
ann5_data = data['occurrence_5']
print(data)

                 occurrence_1      occurrence_2         occurrence_3  \
0                           5           2, 3, 5        1, 3, 4, 6, 8   
1                     1, 3, 4                 5                    1   
2                           4                 7     6, 5, 4, 3, 1, 2   
3                           5  1, 3, 4, 5, 7, 0              4, 5, 7   
4                     6, 7, 8     5, 6, 7, 8, 0                    5   
5                     3, 7, 8           7, 6, 0                    5   
6               2, 4, 5, 1, 0                 2  1, 3, 4, 5, 6, 8, 0   
7   1, 2, 3, 4, 5, 6, 7, 8, 0              6, 7     8, 6, 7, 5, 3, 2   
8               1, 2, 4, 5, 6           7, 8, 3              7, 6, 0   
9                        1, 2                 1                 4, 6   
10                       2, 5                 5  6, 5, 4, 3, 2, 1, 0   
11                       2, 4     1, 2, 3, 4, 5                    3   
12                 1, 2, 3, 4     1, 2, 3, 5, 6        1, 2, 3, 

In [85]:
# just for the DD 
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'

data['S'], data['P'], data['V'], data['LLM'] = [['' for i in range(len(data))] for ii in range(4)]
for i in range(len(df)): 
    for j in range(len(data)): 
        if df['KD ID'][i] == unique_kds[j]: 
           # print(df2['KD ID'][i])
           # print(data['occurrence_3'][j])
            df['DD'][i] = data['occurrence_3'][j]
            data['S'][j] = df['S'][i]
            data['P'][j] = df['P'][i]
            data['V'][j] = df['V'][i]
            data['LLM'][j] = df['LLM'][i]

print(data)

                 occurrence_1      occurrence_2         occurrence_3  \
0                           5           2, 3, 5        1, 3, 4, 6, 8   
1                     1, 3, 4                 5                    1   
2                           4                 7     6, 5, 4, 3, 1, 2   
3                           5  1, 3, 4, 5, 7, 0              4, 5, 7   
4                     6, 7, 8     5, 6, 7, 8, 0                    5   
5                     3, 7, 8           7, 6, 0                    5   
6               2, 4, 5, 1, 0                 2  1, 3, 4, 5, 6, 8, 0   
7   1, 2, 3, 4, 5, 6, 7, 8, 0              6, 7     8, 6, 7, 5, 3, 2   
8               1, 2, 4, 5, 6           7, 8, 3              7, 6, 0   
9                        1, 2                 1                 4, 6   
10                       2, 5                 5  6, 5, 4, 3, 2, 1, 0   
11                       2, 4     1, 2, 3, 4, 5                    3   
12                 1, 2, 3, 4     1, 2, 3, 5, 6        1, 2, 3, 

In [92]:
kappas = [] 
for i in range(5): 
    annots = []
    for idx, row in data.iterrows():
        annot_id = str.zfill(str(idx), 3)
        annot_coder1 = ['S', annot_id, create_annot(str(row[5]))]
        annot_coder2 = ['ann', annot_id, create_annot(str(row[i]))]
        annots.append(annot_coder1)
        annots.append(annot_coder2)
    
    # based on https://stackoverflow.com/questions/45741934/
    task = agreement.AnnotationTask(distance=jaccard_distance)
    task.load_array(annots)
    kappas.append(task.kappa())
print("S-DD", np.mean(kappas))

kappas = [] 
for i in range(5): 
    annots = []
    for idx, row in data.iterrows():
        annot_id = str.zfill(str(idx), 3)
        annot_coder1 = ['V', annot_id, create_annot(str(row[6]))]
        annot_coder2 = ['ann', annot_id, create_annot(str(row[i]))]
        annots.append(annot_coder1)
        annots.append(annot_coder2)
    
    # based on https://stackoverflow.com/questions/45741934/
    task = agreement.AnnotationTask(distance=jaccard_distance)
    task.load_array(annots)
    kappas.append(task.kappa())
print("P-DD", np.mean(kappas))

kappas = [] 
for i in range(5): 
    annots = []
    for idx, row in data.iterrows():
        annot_id = str.zfill(str(idx), 3)
        annot_coder1 = ['S', annot_id, create_annot(str(row[7]))]
        annot_coder2 = ['ann', annot_id, create_annot(str(row[i]))]
        annots.append(annot_coder1)
        annots.append(annot_coder2)
    
    # based on https://stackoverflow.com/questions/45741934/
    task = agreement.AnnotationTask(distance=jaccard_distance)
    task.load_array(annots)
    kappas.append(task.kappa())
print("V-DD", np.mean(kappas))

kappas = [] 
for i in range(5): 
    annots = []
    for idx, row in data.iterrows():
        annot_id = str.zfill(str(idx), 3)
        annot_coder1 = ['S', annot_id, create_annot(str(row[8]))]
        annot_coder2 = ['ann', annot_id, create_annot(str(row[i]))]
        annots.append(annot_coder1)
        annots.append(annot_coder2)
    
    # based on https://stackoverflow.com/questions/45741934/
    task = agreement.AnnotationTask(distance=jaccard_distance)
    task.load_array(annots)
    kappas.append(task.kappa())
print("AI-DD", np.mean(kappas))


S-DD 0.1472149980243722
P-DD 0.2095634861896804
V-DD 0.20544046399884527
AI-DD 0.21539751573204904


In [93]:
print(data.head())

  occurrence_1      occurrence_2      occurrence_3      occurrence_4  \
0            5           2, 3, 5     1, 3, 4, 6, 8              3, 5   
1      1, 3, 4                 5                 1        1, 2, 4, 5   
2            4                 7  6, 5, 4, 3, 1, 2  8, 6, 7, 0, 1, 5   
3            5  1, 3, 4, 5, 7, 0           4, 5, 7                 4   
4      6, 7, 8     5, 6, 7, 8, 0                 5              5, 3   

  occurrence_5  S      P  V LLM  
0      6, 1, 7  0  3,4,5  5   5  
1         1, 4  1      1  1   1  
2            5  0      0  5   7  
3      1, 2, 3  4  4,0,1  4   4  
4            7  0      0  0   7  


In [99]:
# compute both LLM correspondence 
# and do the control group correctly 
import re
from itertools import combinations

def extract_numbers(text):
    """Extract all numbers from a string as a set"""
    return set(re.findall(r'\b\d+(?:\.\d+)?\b', text))

def count_positional_overlaps(lists_of_strings, list_names):
    """
    Count how many positions have overlapping numbers between pairs of lists
    
    Args:
        lists_of_strings: List of 4 lists containing strings in order: 
                         llm_data, S_data, V_data, swits_data
    """
   
    
    # Verify all lists have the same length
    list_lengths = [len(lst) for lst in lists_of_strings]
    if len(set(list_lengths)) > 1:
        raise ValueError("All lists must have the same length")
    
    n_samples = len(lists_of_strings[0])
    
    # Precompute number sets for all positions
    number_sets = []
    for lst in lists_of_strings:
        list_sets = [extract_numbers(s) for s in lst]
        number_sets.append(list_sets)
    
    results = {}
    
    # Check all pair-wise combinations of lists
    for i, j in combinations(range(len(lists_of_strings)), 2):
        count = 0
        for pos in range(n_samples):
            set_i = number_sets[i][pos]
            set_j = number_sets[j][pos]
            
            # Check if both strings have numbers and there's overlap
            if set_i and set_j and set_i.intersection(set_j):
                count += 1
        
        pair_name = f"{list_names[i]}_vs_{list_names[j]}"
        results[pair_name] = count
    
    return results

for ii in ['S', 'P', 'V', 'LLM']:
    print(ii)
    list_names = ['occurrence_1', 'occurrence_2','occurrence_3','occurrence_4','occurrence_5', ii]
    
    llm_data = [str(x) for x in data['occurrence_1'].tolist()]
    S_data = [str(x) for x in data['occurrence_2'].tolist()]
    V_data = [str(x) for x in data['occurrence_3'].tolist()]
    V1_data = [str(x) for x in data['occurrence_4'].tolist()]
    V2_data = [str(x) for x in data['occurrence_5'].tolist()]
    swits_data = [str(x) for x in data[ii].tolist()]
    lists = [llm_data, S_data, V_data,V1_data,V2_data, swits_data]
    result = count_positional_overlaps(lists, list_names)
    
    print("Average position-wise overlapping counts, SWITS:")
    #print("=" * 50)
    sums = [] 
    for i, (pair, count) in enumerate(result.items()):
     #   print(i)
     #   print(f"{pair}: {100*count/len(llm_data)} positions")
        if i == 4 or i==8 or i==11 or i==13 or i==14: 
            sums.append(100*count/len(llm_data))
    print('score: ',np.mean(sums)) 

S
Average position-wise overlapping counts, SWITS:
score:  44.0
P
Average position-wise overlapping counts, SWITS:
score:  57.33333333333333
V
Average position-wise overlapping counts, SWITS:
score:  60.0
LLM
Average position-wise overlapping counts, SWITS:
score:  60.0


In [104]:
# now compute the rest of positional overlaps 
print(df)

    KD ID                                                 KD  KAs (mapping)  \
0   K0640  Knowledge of the organizational cybersecurity ...              7   
1   K0643         Knowledge of virtual learning environments              0   
2   K0645  Knowledge of standard operating procedures (SOPs)              7   
3   K0662          Knowledge of systems security engineering              4   
4   K0706  Knowledge of database schema capabilities and ...              0   
5   K0710  Knowledge of enterprise cybersecurity architec...              7   
6   K0713              Knowledge of Wide Area Networks (WAN)              4   
7   K0719  Knowledge of human-computer interaction (HCI) ...              0   
8   K0732  Knowledge of intrusion detection tools and tec...              4   
9   K0733  Knowledge of information technology (IT) archi...              0   
10  K0757    Knowledge of system design tools and techniques              3   
11  K0765  Knowledge of software engineering princip

In [109]:
list_names = ['LLM', 'S', 'V', 'P'] 
llm_data = [str(x) for x in df['LLM'].tolist()]
S_data = [str(x) for x in df['S'].tolist()]
V_data = [str(x) for x in df['V'].tolist()]
P_data = [str(x) for x in df['P'].tolist()]

lists = [llm_data, S_data, V_data,P_data]
result = count_positional_overlaps(lists, list_names)

print("Average position-wise overlapping counts, SWITS:")
sums = [] 
for i, (pair, count) in enumerate(result.items()):
    #print(count)
    print(f"{pair}: {100*count/len(llm_data)} positions")

Average position-wise overlapping counts, SWITS:
LLM_vs_S: 56.0 positions
LLM_vs_V: 56.0 positions
LLM_vs_P: 46.0 positions
S_vs_V: 52.0 positions
S_vs_P: 48.0 positions
V_vs_P: 48.0 positions


In [113]:
S_data = [str(x) for x in df['S'].tolist()]
V_data = [str(x) for x in df['V'].tolist()]
P_data = [str(x) for x in df['P'].tolist()]
def blub(list1, list2): 
    def digits(s): 
        return set(filter(str.isdigit, s)) 

    count = 0 
    for a,b in zip(list1, list2): 
        if digits(a) & digits(b): 
            count+=1 
    return count 
print(V_data)
print(S_data)
print(blub(llm_data, S_data))

['7', '6', '7', '5', '0', '5, 7', '4', '0', '5', '0', '0', '0', '0', '0', '0', '0', '0', '0', '2', '5', '0', '5', '5', '5', '7', '1', '0', '0', '7', '5', '5', '0', '7', '1', '4', '7', '4', '6', '6', '3', '0', '0', '5', '0', '7', '5', '5', '5', '7', '7']
['7', '0', '7', '4', '0', '7', '4', '0', '4', '0', '3', '2', '0', '0', '0', '0', '0', '0', '2', '7', '0', '0', '0', '4', '7', '0', '0', '0', '0', '0', '0', '7', '0', '1', '4', '7', '4', '0', '8', '6', '5', '2', '0', '0', '0', '7', '5', '0', '7', '7']
28


In [107]:
print(len(S_data))
print(len(llm_data))

50
50
